# A: pooled direct logistic validation

One model per lead; all oblasts pooled, no identity encoding, calendar or STL predictors. Leads 1–6 are primary; lead 0 is separate; 12/24 are secondary. Each fold starts from scratch on TRAIN ending 25 hours before VALID. Scaling, rates and coefficients fit TRAIN only. C is selected by pooled VALID Brier over four independent folds; these are selection scores, not unbiased final performance. No resampling or TEST evaluation.


In [1]:
from pathlib import Path
import sys
import pandas as pd
root = Path.cwd() if Path("src/airalerts").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))
from airalerts.evaluate import latest_results
out = root / "reports/model"
latest = latest_results(out / "results.csv")


In [2]:
results = pd.read_csv(out / "A_validation.csv", dtype={"fold_id": str})
summary = results.loc[results.fold_id.eq("pooled") & results.scope.eq("ALL")]
print(summary[["model", "lead", "n", "brier", "bss_vs_clim", "bss_vs_persistence", "bss_vs_ses", "bss_ci_low", "bss_ci_high", "ece"]].to_string(index=False))
print(pd.read_csv(out / "A_selected.csv").to_string(index=False))


   model  lead     n    brier  bss_vs_clim  bss_vs_persistence  bss_vs_ses  bss_ci_low  bss_ci_high      ece
logreg_A     0 35136 0.055035     0.436156            0.038825    0.346231    0.408764     0.464261 0.013652
logreg_A     1 35112 0.088828     0.090061            0.031964    0.034661    0.061513     0.125117 0.028523
logreg_A     2 35088 0.092313     0.054536            0.034285    0.011749    0.035037     0.074399 0.024917
logreg_A     3 35064 0.093048     0.047078            0.032443    0.002618    0.030935     0.063320 0.024906
logreg_A     4 35040 0.093455     0.042923            0.027614   -0.009699    0.026645     0.058107 0.022582
logreg_A     5 35016 0.094006     0.037034            0.023344   -0.013741    0.021355     0.051323 0.022965
logreg_A     6 34992 0.094629     0.030693            0.020117   -0.018716    0.013141     0.045802 0.024214
logreg_A    12 34848 0.096008     0.017121            0.014159   -0.028945   -0.008072     0.037421 0.028813
logreg_A    24 3456

In [3]:
reference = pd.read_csv(out / "reference_validation.csv")
print(reference.loc[reference.scope.eq("ALL") & reference.model.isin(["climatology", "persistence_prob", "ses"]), ["model", "lead", "brier", "bss_vs_clim", "ece"]].to_string(index=False))


           model  lead    brier  bss_vs_clim      ece
     climatology     0 0.097607     0.000000 0.044374
     climatology     1 0.097620     0.000000 0.044424
     climatology     2 0.097637     0.000000 0.044419
     climatology     3 0.097645     0.000000 0.044389
     climatology     4 0.097646     0.000000 0.044435
     climatology     5 0.097621     0.000000 0.044475
     climatology     6 0.097625     0.000000 0.044500
     climatology    12 0.097680     0.000000 0.044381
     climatology    24 0.097126     0.000000 0.043809
persistence_prob     0 0.057258     0.413381 0.021551
persistence_prob     1 0.091761     0.060016 0.032206
persistence_prob     2 0.095590     0.020971 0.039421
persistence_prob     3 0.096168     0.015126 0.037585
persistence_prob     4 0.096109     0.015743 0.038623
persistence_prob     5 0.096253     0.014018 0.037428
persistence_prob     6 0.096571     0.010793 0.038462
persistence_prob    12 0.097387     0.003005 0.039562
persistence_prob    24 0.096

## Calibration

ECE uses ten bins. CIs pair all regions within circular seven-day UTC blocks; one-day blocks are sensitivity only. Flags are computed separately from each fold TRAIN target rate, below 3% or above 95%; per-oblast and pooled excluded-flagged metrics are in results.csv.


![Calibration: A](../reports/model/A_calibration.png)

Pooled validation BSS versus climatology is 0.09006 at lead 1 and 0.03069 at lead 6. At lead 1 BSS versus persistence is 0.03196 and versus SES is 0.03466. Longer-lead estimates and confidence intervals are in the table; selection used VALID and no final-test conclusion is drawn.
